# Disinformation Detection Pipeline — Demo on RU22Fact

This notebook runs a stratified sample of the **RU22Fact** training corpus through the full `DisinformationDetectionPipeline` and visualises how well the pipeline's verdicts align with the dataset ground-truth labels.

## Prerequisites

1. **Ollama** must be running locally (`ollama serve`) with the configured model (default: `llama3`).
2. **`PINECONE_API_KEY`** must be set and the `ru22fact-evidence` index must already be populated (run `disinfo-index` first).
3. Extra Python deps — install once:
   ```
   pip install matplotlib seaborn scikit-learn tqdm jupyter ipykernel
   ```


In [ ]:
import sys
import pathlib

from pandas import DataFrame, Series

# Resolve project root relative to this notebook (notebooks/ is one level below root)
PROJECT_ROOT = pathlib.Path.cwd().resolve().parent
if not (PROJECT_ROOT / "pipeline.py").exists():
    # Fallback: absolute path for cases where Jupyter was launched from a different cwd
    PROJECT_ROOT = pathlib.Path(r"D:\Programming\diploma")
if not (PROJECT_ROOT / "pipeline.py").exists():
    raise RuntimeError(
        f"Cannot locate project root (tried {PROJECT_ROOT}). "
        "Launch Jupyter from D:\\Programming\\diploma\\notebooks or update PROJECT_ROOT above."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
import json
import time
import traceback

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    classification_report,
)
from tqdm.auto import tqdm

from explainability.explainer import Explainer
from pipeline import DisinformationDetectionPipeline

sns.set_theme(style="whitegrid")
print("Imports OK")

## 1 · Configuration

Adjust `SAMPLE_SIZE` and `FILTER_NEI` here; everything else follows.

In [ ]:
DATASET_PATH     = PROJECT_ROOT / "data" / "datasets" / "ru22fact" / "ru22fact_train.csv"
RESULTS_DIR      = PROJECT_ROOT / "notebooks" / "results"
SUMMARY_CSV      = RESULTS_DIR / "summary.csv"

SAMPLE_SIZE      = 2    # number of claims to verify (stratified by language + label)
FILTER_NEI       = True   # drop NEI rows before sampling
RANDOM_SEED      = 42
CHECKPOINT_EVERY = 25     # write summary.csv every N rows
RESUME           = True   # skip claims whose <id>.json already exists in RESULTS_DIR

LANGUAGE_MAP = {
    "English":   "en",
    "Ukrainian": "uk",
    "Russian":   "ru",
    "Chinese":   "zh",
}

# Dataset label → expected pipeline verdict
LABEL_TO_VERDICT = {"Supported": "CONFIRMED", "Refuted": "DISINFORMATION"}

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Results dir: {RESULTS_DIR}")

## 2 · Health check

The notebook aborts here with a clear remediation message if Pinecone or Ollama is unavailable.

In [ ]:
pipeline = DisinformationDetectionPipeline()
status = pipeline.health_check()
print("Service status:", status)

if not status["pinecone"]:
    raise RuntimeError(
        "Pinecone is unreachable. Check that PINECONE_API_KEY is set and the "
        "'ru22fact-evidence' index exists, then re-run this cell."
    )
if not status["ollama"]:
    raise RuntimeError(
        "Ollama is unreachable. Start it with `ollama serve` (default port 11434) "
        "and ensure the configured model is pulled, then re-run this cell."
    )

print("✓ All services reachable.")

## 3 · Load dataset and draw a stratified sample

In [ ]:
df = pd.read_csv(DATASET_PATH)
print(f"Loaded {len(df):,} rows — columns: {list(df.columns)}")

# Drop rows with missing claim, label, or language
df = df.dropna(subset=["claim", "label", "language"])
# Keep only languages the pipeline can handle
df = df[df["language"].isin(LANGUAGE_MAP)]

if FILTER_NEI:
    df = df[df["label"].isin(LABEL_TO_VERDICT)]
    print(f"After filtering NEI: {len(df):,} rows remain")

print("\nLabel × Language distribution:")
display(df.groupby(["language", "label"]).size().unstack(fill_value=0))

In [ ]:
def stratified_sample(df: pd.DataFrame, n: int, seed: int) -> DataFrame | Series:
    """Proportional stratified sample by (language, label) with per-stratum floor of 1.

    Residual adjustment ensures the total stays within 1 of the requested n,
    even when some strata are very small.
    """
    rng = np.random.default_rng(seed)
    strata = df.groupby(["language", "label"])
    total = len(df)

    quotas: dict = {}
    for key, sub in strata:
        raw = max(1, int(round(len(sub) / total * n)))
        quotas[key] = min(raw, len(sub))

    # Adjust sum to match n
    diff = n - sum(quotas.values())
    keys_sorted = sorted(quotas, key=lambda k: quotas[k], reverse=True)
    i = 0
    max_iters = 10 * len(keys_sorted) + 1
    while diff != 0 and i < max_iters:
        k = keys_sorted[i % len(keys_sorted)]
        stratum_size = len(strata.get_group(k))
        if diff > 0 and quotas[k] < stratum_size:
            quotas[k] += 1
            diff -= 1
        elif diff < 0 and quotas[k] > 1:
            quotas[k] -= 1
            diff += 1
        i += 1

    parts = [
        strata.get_group(k).sample(n=q, random_state=int(rng.integers(0, 2**31)))
        for k, q in quotas.items() if q > 0
    ]
    return pd.concat(parts).sample(frac=1.0, random_state=seed).reset_index(drop=True)


sample_df = stratified_sample(df, SAMPLE_SIZE, RANDOM_SEED)
sample_df["lang_code"] = sample_df["language"].map(LANGUAGE_MAP)

print(f"Sampled {len(sample_df)} claims")
print("\nSample Label × Language distribution:")
display(sample_df.groupby(["language", "label"]).size().unstack(fill_value=0))

## 4 · Initialize pipeline

This loads the NLI model (cross-encoder/nli-deberta-v3-base), the embedding model,
and establishes the Pinecone connection. Expect **30–90 seconds** on first run.

In [ ]:
try:
    pipeline.initialize()
    print("Pipeline initialized.")
except Exception as exc:
    print(f"Initialization failed ({type(exc).__name__}): {exc}")
    raise

## 5 · Inference loop

Each claim is verified individually via `analyze_single_claim`. Results are:
- written to `results/<id>.json` immediately after each call.
- summarised in `results/summary.csv` every `CHECKPOINT_EVERY` rows.

If `RESUME=True`, claims whose JSON already exists are reloaded from disk rather than re-queried.

**Estimated time:** ~3–10 s/claim → 10–30 min for N=200.

In [ ]:
explainer = Explainer()
records: list[dict] = []


def _build_record(
    row: dict,
    output,
    latency: float | None,
    error: str | None,
    resumed: bool,
) -> dict:
    return {
        "id":              row["id"],
        "claim":           row["claim"],
        "gold_label":      row["label"],
        "language":        row["language"],
        "lang_code":       row["lang_code"],
        "verdict":         getattr(output, "verdict", None),
        "final_score":     getattr(output, "final_score", None),
        "nli_score":       (output.component_scores or {}).get("nli") if output else None,
        "rag_score":       (output.component_scores or {}).get("rag") if output else None,
        "evidence_count":  len(output.evidence_excerpts) if output else 0,
        "latency_seconds": latency,
        "error":           error,
        "resumed":         resumed,
    }


def _checkpoint(records: list[dict]) -> None:
    pd.DataFrame.from_records(records).to_csv(SUMMARY_CSV, index=False)


for i, row in enumerate(tqdm(sample_df.to_dict("records"), desc="Verifying claims")):
    dest = RESULTS_DIR / f"{row['id']}.json"

    # --- Resume from disk -------------------------------------------------
    if RESUME and dest.exists():
        try:
            raw = json.loads(dest.read_text(encoding="utf-8"))
            # Reconstruct a minimal object for _build_record
            class _Stub:
                pass
            output = _Stub()
            output.verdict = raw.get("verdict")
            output.final_score = raw.get("final_score")
            output.component_scores = raw.get("component_scores", {})
            output.evidence_excerpts = raw.get("evidence_excerpts", [])
            records.append(_build_record(row, output, None, None, resumed=True))
            continue
        except Exception:
            pass  # corrupted JSON — re-run the claim

    # --- Live inference ---------------------------------------------------
    output, error, latency = None, None, None
    start = time.perf_counter()
    try:
        output = pipeline.analyze_single_claim(row["claim"], language=row["lang_code"])
        dest.write_text(explainer.to_json(output), encoding="utf-8")
    except Exception as exc:
        error = f"{type(exc).__name__}: {exc}"
        (RESULTS_DIR / f"{row['id']}.error.txt").write_text(
            traceback.format_exc(), encoding="utf-8"
        )
    finally:
        latency = time.perf_counter() - start

    records.append(_build_record(row, output, latency, error, resumed=False))

    if (i + 1) % CHECKPOINT_EVERY == 0:
        _checkpoint(records)

_checkpoint(records)
results_df = pd.read_csv(SUMMARY_CSV)
print(f"\nDone. {len(results_df)} rows in summary.")
print(f"Errors: {results_df['error'].notna().sum()}")
print(f"Resumed from disk: {results_df['resumed'].sum()}")
results_df.head()

## 6 · Performance metrics

We report **two parallel metric sets** to give an honest picture:

| Metric set | What it measures | UNCERTAIN treatment |
|---|---|---|
| **Decided-only** | Accuracy when the system commits to a verdict | Excluded from denominator |
| **Penalised** | End-to-end accuracy (downstream consumer view) | Counted as wrong |

The **abstention rate** is always shown alongside — a high abstention rate trivially inflates decided-only numbers.

The pipeline's aggregator deliberately emits `UNCERTAIN` when NLI and RAG strongly disagree (see `verification/aggregator.py:113`). Silently collapsing that to "wrong" or "Refuted" would misrepresent that intentional abstention signal.

In [ ]:
# Keep only successful calls
ok = results_df[
    results_df["error"].isna() | (results_df["error"] == "")
].copy()

# Map pipeline verdict to human-readable predicted label
ok["predicted_label"] = ok["verdict"].map({
    "CONFIRMED":     "Supported",
    "DISINFORMATION": "Refuted",
    "UNCERTAIN":      "Uncertain",
})

labels_gold = ["Supported", "Refuted"]
labels_pred = ["Supported", "Refuted", "Uncertain"]

# 3×2 confusion matrix
cm = pd.crosstab(ok["predicted_label"], ok["gold_label"]).reindex(
    index=labels_pred, columns=labels_gold, fill_value=0
)
print("Confusion matrix (rows=predicted, cols=gold):")
display(cm)

# --- Decided-only metrics (UNCERTAIN excluded) ---
decided = ok[ok["predicted_label"].isin(labels_gold)].copy()
abstain_rate = 1 - len(decided) / max(len(ok), 1)

if len(decided) > 0:
    report_decided = classification_report(
        decided["gold_label"],
        decided["predicted_label"],
        labels=labels_gold,
        zero_division=0,
        output_dict=True,
    )
    print("\nDecided-only classification report:")
    print(classification_report(
        decided["gold_label"],
        decided["predicted_label"],
        labels=labels_gold,
        zero_division=0,
    ))
else:
    report_decided = {"accuracy": 0.0, "macro avg": {"f1-score": 0.0}}
    print("No decided predictions to evaluate.")

# --- Penalised metrics (UNCERTAIN = wrong) ---
penalised_pred = ok["predicted_label"].where(
    ok["predicted_label"].isin(labels_gold), other="__abstain__"
)
acc_penalised = accuracy_score(ok["gold_label"], penalised_pred)

print(f"\nAbstention rate:       {abstain_rate:.2%}")
print(f"Penalised accuracy:    {acc_penalised:.3f}")

## 7 · Visualizations

In [ ]:
# 7.1 Confusion matrix heatmap
fig, ax = plt.subplots(figsize=(7, 4))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    linewidths=0.5, ax=ax,
)
ax.set_title("Verdict vs. Ground Truth (rows = predicted, cols = gold)")
ax.set_xlabel("Gold label")
ax.set_ylabel("Predicted label")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "confusion_matrix.png", dpi=150)
plt.show()

In [ ]:
# 7.2 Accuracy by language (decided-only)
if len(decided) > 0:
    lang_acc = (
        decided
        .groupby("language")
        .apply(lambda g: pd.Series({
            "accuracy": (g["predicted_label"] == g["gold_label"]).mean(),
            "count":    len(g),
        }))
        .sort_values("accuracy", ascending=False)
    )

    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.bar(lang_acc.index, lang_acc["accuracy"], color=sns.color_palette("Blues_d", len(lang_acc)))
    for bar, (_, row) in zip(bars, lang_acc.iterrows()):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01,
            f"n={int(row['count'])}",
            ha="center", va="bottom", fontsize=9,
        )
    ax.set_ylim(0, 1.15)
    ax.set_title("Decided-only Accuracy by Language")
    ax.set_xlabel("Language")
    ax.set_ylabel("Accuracy")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "accuracy_by_language.png", dpi=150)
    plt.show()
else:
    print("No decided predictions — skipping language accuracy chart.")

In [ ]:
# 7.3 Final-score distribution split by gold label
fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(
    data=ok.dropna(subset=["final_score"]),
    x="final_score",
    hue="gold_label",
    element="step",
    stat="density",
    kde=True,
    alpha=0.4,
    ax=ax,
)
ax.set_title("Final Score Distribution by Gold Label")
ax.set_xlabel("Final truthfulness score (0 = disinformation, 1 = confirmed)")
ax.set_ylabel("Density")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "score_distribution.png", dpi=150)
plt.show()

In [ ]:
# 7.4 NLI vs RAG component scores
scatter_df = ok.dropna(subset=["nli_score", "rag_score"])
if len(scatter_df) > 0:
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.scatterplot(
        data=scatter_df,
        x="nli_score", y="rag_score",
        hue="gold_label", style="verdict",
        alpha=0.7, s=60, ax=ax,
    )
    # y=x reference
    lims = [0, 1]
    ax.plot(lims, lims, "--", color="gray", linewidth=0.8, label="NLI = RAG")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_title("NLI Score vs RAG Score (per claim)")
    ax.set_xlabel("NLI score")
    ax.set_ylabel("RAG score")
    ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "nli_vs_rag_scatter.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No NLI/RAG scores to plot.")

In [ ]:
# 7.5 Latency histogram + percentiles
latencies = results_df["latency_seconds"].dropna()
if len(latencies) > 0:
    p50 = np.percentile(latencies, 50)
    p95 = np.percentile(latencies, 95)

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(latencies, bins=30, color=sns.color_palette("Blues")[3], edgecolor="white")
    ax.axvline(p50, color="navy",  linestyle="--", label=f"p50 = {p50:.1f}s")
    ax.axvline(p95, color="crimson", linestyle="--", label=f"p95 = {p95:.1f}s")
    ax.set_title("Per-claim Latency Distribution")
    ax.set_xlabel("Latency (seconds)")
    ax.set_ylabel("Count")
    ax.legend()
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "latency_histogram.png", dpi=150)
    plt.show()
    print(f"p50 = {p50:.2f}s  |  p95 = {p95:.2f}s")
else:
    print("No latency data (all rows resumed from disk).")

In [ ]:
# 7.6 Verdict distribution
verdict_counts = ok["verdict"].value_counts()
fig, ax = plt.subplots(figsize=(6, 4))
colors = sns.color_palette("Set2", len(verdict_counts))
verdict_counts.plot.bar(ax=ax, color=colors, edgecolor="white")
ax.set_title("Pipeline Verdict Distribution")
ax.set_xlabel("Verdict")
ax.set_ylabel("Count")
ax.tick_params(axis="x", rotation=0)
for patch in ax.patches:
    ax.text(
        patch.get_x() + patch.get_width() / 2,
        patch.get_height() + 0.3,
        str(int(patch.get_height())),
        ha="center", va="bottom",
    )
plt.tight_layout()
plt.savefig(RESULTS_DIR / "verdict_distribution.png", dpi=150)
plt.show()

## 8 · Summary

In [ ]:
latencies_live = results_df["latency_seconds"].dropna()
mean_evidence  = ok["evidence_count"].mean() if len(ok) > 0 else float("nan")

print("=" * 52)
print("  Disinformation Detection Pipeline — Demo Summary")
print("=" * 52)
print(f"  Sample size:               {len(results_df)}")
print(f"  Successful calls:          {len(ok)}")
print(f"  Failed calls:              {results_df['error'].notna().sum()}")
print(f"  Resumed from disk:         {int(results_df['resumed'].sum())}")
print(f"  Abstention rate:           {abstain_rate:.2%}")
print(f"  Decided-only accuracy:     {report_decided.get('accuracy', 0):.3f}")
print(f"  Decided-only macro F1:     {report_decided.get('macro avg', {}).get('f1-score', 0):.3f}")
print(f"  Penalised accuracy:        {acc_penalised:.3f}")
if len(latencies_live) > 0:
    print(f"  Latency p50 / p95 (s):     {np.percentile(latencies_live, 50):.2f} / {np.percentile(latencies_live, 95):.2f}")
else:
    print("  Latency:                   (all resumed — no live calls)")
print(f"  Mean evidence per claim:   {mean_evidence:.1f}")
if mean_evidence < 1:
    print("  ⚠️  Mean evidence < 1 — Pinecone index may be empty or misconfigured.")
print(f"  Artifacts:                 {RESULTS_DIR}")
print("=" * 52)